# Prerequisite 04 — LFW aligned crop materialization

LFW manifest를 공통 ArcFace 112×112 RGB crop으로 정렬합니다. 이 결과는 모든 Step 2 checkpoint가 공유하며, 검출 실패를 center crop으로 대체하지 않습니다.

**순서:** 이 노트북은 dataset 00 이후, model/embedding 및 Grad-CAM 노트북 이전에 한 번 실행합니다.

In [ ]:
from pathlib import Path
import sys
import yaml

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "pyproject.toml").is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("pyproject.toml을 찾을 수 없습니다.")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs/experiments/step2_pytorch_gradcam.yaml"
CONFIG = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))

In [ ]:
DATA_FRACTION = 1.0
EXECUTE_STAGE = True
WRITE_OUTPUTS = True
OVERWRITE = True
SEED = 42

if DATA_FRACTION != 1.0:
    raise ValueError("공통 aligned crop은 DATA_FRACTION=1.0이어야 합니다.")
if WRITE_OUTPUTS and not EXECUTE_STAGE:
    raise ValueError("WRITE_OUTPUTS=True이면 EXECUTE_STAGE도 True여야 합니다.")

In [ ]:
import pandas as pd

from research.preprocessing import materialize_aligned_crops

SOURCE_MANIFEST_PATH = PROJECT_ROOT / CONFIG["datasets"]["lfw"]["manifest_path"]
OUTPUT_DIR = PROJECT_ROOT / CONFIG["aligned_crops"]["bundle_dir"]

In [ ]:
if EXECUTE_STAGE:
    if not SOURCE_MANIFEST_PATH.is_file():
        raise FileNotFoundError(
            "먼저 notebooks/lfw/00_data_preparation/00_data_preparation.ipynb를 "
            f"실행하세요: {SOURCE_MANIFEST_PATH}"
        )
    source_manifest = pd.read_csv(SOURCE_MANIFEST_PATH)
    result = materialize_aligned_crops(
        source_manifest,
        project_root=PROJECT_ROOT,
        output_dir=OUTPUT_DIR,
        dataset_id="lfw",
        overwrite=OVERWRITE,
    )
    summary = {
        "output_dir": str(result.output_dir),
        **result.bundle_manifest["counts"],
        "array_contract": result.bundle_manifest["array_contract"],
    }
else:
    summary = {"status": "not_executed"}
summary

## 저장 계약

- `aligned_faces.npy`: 모델 입력용 binary array
- `aligned_index.csv`: 사람이 확인 가능한 sample↔array index
- `failed_samples.csv`: 실패 사유 전수
- `bundle_manifest.json`, `_SUCCESS`: hash·shape·완료 상태

`OVERWRITE=True`는 완성된 대체 bundle을 먼저 staging한 뒤 canonical 폴더 하나만 교체합니다. 이를 사용한 완료 run은 기록된 입력 hash와 함께 계속 불변입니다.